# RAG System for Web Data — Step-by-step Walkthrough

This notebook walks through every component of the RAG pipeline:
1. **Ingestion** — fetch & chunk web pages
2. **Embeddings** — convert text to vectors
3. **Vectorstore** — store & search with ChromaDB
4. **Retrieval** — find the most relevant chunks
5. **Generation** — answer questions with an LLM

> **Requirements**: Ollama running locally with `llama3` and `nomic-embed-text` pulled.
> ```bash
> ollama pull llama3
> ollama pull nomic-embed-text
> ```

## 0. Setup

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

from src.config import load_config

config = load_config('../config.yaml')

print('URLs to index:')
for url in config.urls:
    print(f'  - {url}')
print(f'\nChunk size: {config.chunking.chunk_size} chars, overlap: {config.chunking.chunk_overlap}')
print(f'LLM: {config.llm.provider} / {config.llm.model}')
print(f'Embeddings: {config.embeddings.provider} / {config.embeddings.model}')

---
## 1. Ingestion — Fetch & Chunk Web Pages

We fetch content from URLs, strip HTML, and split into overlapping chunks.
The `chunk_overlap` ensures context isn't lost at chunk boundaries.

In [ ]:
from src.ingestion import WebIngestion

ingestor = WebIngestion(config)

# Load raw documents
documents = ingestor.load_urls()

print(f'\nLoaded {len(documents)} documents.')
print(f'\nSample (first 400 chars):')
print(documents[0].page_content[:400])
print(f'\nMetadata: {documents[0].metadata}')

In [ ]:
# Split into chunks
chunks = ingestor.split_documents(documents)

print(f'Total chunks: {len(chunks)}')
print(f'Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')
print(f'\nSample chunk:')
print(chunks[10].page_content)
print(f'Metadata: {chunks[10].metadata}')

---
## 2. Embeddings — Convert Text to Vectors

Each chunk is converted to a dense vector representation.
Similar texts will have vectors that are close in the embedding space.

In [ ]:
from src.embeddings import get_embeddings
import numpy as np

embeddings = get_embeddings(config)

# Embed a sample sentence to inspect the vector
sample_text = 'What is Retrieval-Augmented Generation?'
vector = embeddings.embed_query(sample_text)

print(f'Embedding dimension: {len(vector)}')
print(f'First 8 values: {[round(v, 4) for v in vector[:8]]}')

In [ ]:
# Demonstrate semantic similarity
from numpy.linalg import norm

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (norm(a) * norm(b)))

v1 = embeddings.embed_query('What is RAG?')
v2 = embeddings.embed_query('Retrieval-Augmented Generation explained')
v3 = embeddings.embed_query('How to cook pasta')

print(f'Similarity (RAG vs RAG explained) : {cosine_similarity(v1, v2):.4f}  ← high')
print(f'Similarity (RAG vs cooking pasta) : {cosine_similarity(v1, v3):.4f}  ← low')

---
## 3. Vectorstore — Index Chunks with ChromaDB

All chunk embeddings are stored in ChromaDB, which supports
fast approximate nearest-neighbour search.

In [ ]:
from src.vectorstore import VectorStore

vs = VectorStore(config, embeddings)
db = vs.build(chunks)  # build and persist to disk

print('Vectorstore built and persisted.')

In [ ]:
# Quick similarity search to verify the index works
results = vs.similarity_search_with_score('What is LangChain?', k=3)

print('Top 3 results for "What is LangChain?"\n')
for doc, score in results:
    print(f'Score (L2): {score:.4f}')
    print(f'Source: {doc.metadata.get("source", "unknown")}')
    print(f'Chunk: {doc.page_content[:200]}...')
    print()

---
## 4. Retriever — Find the Most Relevant Chunks

The retriever wraps the vectorstore and exposes a standard
LangChain interface for use in chains.

In [ ]:
from src.retriever import get_retriever

retriever = get_retriever(db, config)

# Test retrieval
query = 'How does RAG reduce hallucination?'
retrieved_docs = retriever.invoke(query)

print(f'Query: "{query}"')
print(f'Retrieved {len(retrieved_docs)} chunks:\n')
for i, doc in enumerate(retrieved_docs, 1):
    print(f'[{i}] {doc.page_content[:250]}...')
    print(f'    Source: {doc.metadata.get("source")}')
    print()

---
## 5. Generation — Answer Questions with an LLM

The retrieved chunks are injected into the prompt as context.
The LLM is instructed to answer *only* from that context.

In [ ]:
from src.llm import get_llm
from src.chain import RAGChain

llm = get_llm(config)
rag = RAGChain(retriever, llm, config)

print('RAG chain ready.')

In [ ]:
response = rag.ask('What is LangChain?')
print(response)

In [ ]:
response = rag.ask('What are the main limitations of LLMs that RAG addresses?')
print(response)

In [ ]:
response = rag.ask('What is a vector database and why is it used in RAG?')
print(response)

---
## 6. Grounding Test — Out-of-scope Question

A key RAG property: the system should *refuse* to answer questions
that aren't covered by the knowledge base.

In [ ]:
# This should trigger the "I don't have enough information" guard
response = rag.ask('What is the capital of Australia?')
print(response)

---
## 7. Add Your Own URLs

You can index any public URL and immediately query it.

In [ ]:
# Add a new URL to the knowledge base
NEW_URLS = [
    'https://en.wikipedia.org/wiki/Vector_database'
]

new_docs = ingestor.load_urls(NEW_URLS)
new_chunks = ingestor.split_documents(new_docs)

# Add to existing vectorstore
db.add_documents(new_chunks)
print(f'Added {len(new_chunks)} new chunks to the vectorstore.')

In [ ]:
# Now query about the new content
response = rag.ask('What are the main vector database providers?')
print(response)

---
## Summary

| Component | Module | Role |
|-----------|--------|------|
| Ingestion | `src/ingestion.py` | Fetch URLs, clean HTML, split into chunks |
| Embeddings | `src/embeddings.py` | Convert text to dense vectors |
| Vectorstore | `src/vectorstore.py` | Store & search vectors (ChromaDB) |
| Retriever | `src/retriever.py` | Find top-k most relevant chunks |
| LLM | `src/llm.py` | Generate grounded answers |
| Chain | `src/chain.py` | Assemble the full pipeline |

To run from the command line:
```bash
python main.py --question "What is LangChain?"
python main.py --interactive
```